# EVO test

In [ ]:
import pandas as pd
import os

## Load data

In [ ]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

In [ ]:
phages_df.head()

## Embeddings

In [2]:
import torch
from evo import Evo

torch.__version__

'2.1.2+cu121'

In [15]:
# device = 'cuda:0'
device = 'cpu'

In [ ]:
evo_model = Evo('evo-1-8k-base', device=device)
model, tokenizer = evo_model.model, evo_model.tokenizer

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
model.to(device)
model.eval()

StripedHyenaModelForCausalLM(
  (backbone): StripedHyena(
    (embedding_layer): VocabParallelEmbedding(512, 4096)
    (norm): RMSNorm()
    (unembed): VocabParallelEmbedding(512, 4096)
    (blocks): ModuleList(
      (0-7): 8 x ParallelGatedConvBlock(
        (pre_norm): RMSNorm()
        (post_norm): RMSNorm()
        (filter): ParallelHyenaFilter()
        (projections): Linear(in_features=4096, out_features=12288, bias=True)
        (out_filter_dense): Linear(in_features=4096, out_features=4096, bias=True)
        (mlp): ParallelGatedMLP(
          (l1): Linear(in_features=4096, out_features=10928, bias=False)
          (l2): Linear(in_features=4096, out_features=10928, bias=False)
          (l3): Linear(in_features=10928, out_features=4096, bias=False)
        )
      )
      (8): AttentionBlock(
        (pre_norm): RMSNorm()
        (post_norm): RMSNorm()
        (inner_mha_cls): MHA(
          (rotary_emb): RotaryEmbedding()
          (Wqkv): Linear(in_features=4096, out_feature

In [10]:
# monkey patch the unembed function with identity
# this removes the final projection back from the embedding space into tokens
# so the "logits" of the model is now the final layer embedding
# see source for unembed - https://huggingface.co/togethercomputer/evo-1-131k-base/blob/main/model.py#L339

from torch import nn

class CustomEmbedding(nn.Module):
  def unembed(self, u):
    return u

model.unembed = CustomEmbedding()

# end custom code

In [11]:
sequence = 'ACGT'
input_ids = torch.tensor(
    tokenizer.tokenize(sequence),
    dtype=torch.int,
).to(device).unsqueeze(0)

In [12]:
embed, _ = model(input_ids) # (batch, length, embed dim)

print('Embed: ', embed)
print('Shape (batch, length, embed dim): ', embed.shape)

Initializing inference params...


ValueError: Cannot find backend for cpu

In [ ]:
# you can now use embedding for downstream classification tasks
# you probably want to aggregate over position dimension
mean_emb = embed.mean(dim=1)

## Embeddings with huggingface

In [8]:
# Load model directly
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("togethercomputer/evo-1-8k-base", trust_remote_code=True, torch_dtype="auto")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]